# Day I5 — 근육 활동과 EMG: 정류, envelope, 정규화, onset을 직접 구현하기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait/day-I5.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait/day-I5.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 정류하지 않고 평활하면 왜 안 되는가 (가상 신호)

In [ ]:
import numpy as np
from scipy import signal

rng = np.random.default_rng(0)
fs = 1000
t = np.arange(0, 2, 1 / fs)
drive = np.where((t > 0.5) & (t < 1.2), 1.0, 0.1)          # 0.5–1.2 s에만 근육이 '켜짐'(가상)
emg = drive * rng.normal(0, 1, t.size)                      # 진폭이 drive를 따르는 무작위 신호

sos = signal.butter(2, 6, btype="low", fs=fs, output="sos")
lp_raw = signal.sosfiltfilt(sos, emg)                       # 정류 없이 바로 low-pass
lp_rect = signal.sosfiltfilt(sos, np.abs(emg))              # 정류한 뒤 low-pass

on = (t > 0.6) & (t < 1.1)
print("켜진 구간 평균  | 정류 없이: %.3f | 정류 후: %.3f" % (lp_raw[on].mean(), lp_rect[on].mean()))
print("꺼진 구간 평균  | 정류 없이: %.3f | 정류 후: %.3f" % (lp_raw[t < 0.4].mean(), lp_rect[t < 0.4].mean()))

### 예시 2 — BMC `emg.csv`를 처음부터 끝까지 처리하기

> **데이터:** BMC v0.2.0의 `data/emg.csv`(내용 CC BY 4.0, [doi:10.5281/zenodo.23654869](https://doi.org/10.5281/zenodo.23654869)). 머리글 없이 2열(시간 s, 진폭)이고 1000 Hz로 3.36초입니다. **어느 근육, 어떤 과제, 어떤 단위인지 파일에 정보가 없으므로** 오늘은 처리 방법 연습용으로만 쓰고 생리적 해석은 하지 않습니다. BMC의 `electromyography` 노트북은 진폭 정규화를 하지 않고 onset 검출에 외부 패키지(`detecta`)를 쓰는데, 과정 환경에는 그 패키지가 없으므로 아래 코드는 numpy/scipy로 직접 구현했습니다.
>
> ```bash
> mkdir -p data/bmc
> curl -o data/bmc/emg.csv https://raw.githubusercontent.com/BMClab/BMC/master/data/emg.csv
> ```

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

d = np.loadtxt("data/bmc/emg.csv", delimiter=",")          # 1열 시간(s), 2열 EMG(단위·근육 정보 없음)
t, raw = d[:, 0], d[:, 1]
fs = 1 / np.median(np.diff(t))
print("표본 %d개, fs = %.0f Hz, 길이 %.2f s" % (len(t), fs, t[-1]))

# 1) 평균(DC) 제거 → 2) band-pass 20–400 Hz (움직임 잡음은 아래, Nyquist 500 Hz보다 낮게)
x = raw - raw.mean()
sos_bp = signal.butter(4, [20, 400], btype="bandpass", fs=fs, output="sos")
x_bp = signal.sosfiltfilt(sos_bp, x)
# 3) 전파 정류(full-wave rectification)
x_rect = np.abs(x_bp)

# 4) linear envelope: 정류 신호를 low-pass (cutoff를 바꿔 본다)
def envelope(rect, fc):
    sos = signal.butter(2, fc, btype="low", fs=fs, output="sos")
    return signal.sosfiltfilt(sos, rect)                     # 시점 비교가 목적이라 zero-phase

# 5) 진폭 정규화: MVC 기록이 없으므로 '이 기록 안의 최댓값 = 100%'(과제 내 최대 정규화)
# 6) onset/offset: 임계값을 넘는 구간 중 min_dur 이상 지속되는 것만
def bursts(env, thr, min_dur=0.05):
    above = np.r_[False, env > thr, False].astype(int)
    on = np.flatnonzero(np.diff(above) == 1)
    off = np.flatnonzero(np.diff(above) == -1)
    keep = (off - on) / fs >= min_dur
    return t[on[keep]], t[off[keep] - 1]

base = t < 0.3                                               # 처음 0.3 s를 쉬는 구간(baseline)으로 가정
print("\nfc(Hz)  임계값 규칙          burst 수  onset(s)들")
for fc in [3, 6, 10]:
    env = envelope(x_rect, fc)
    env_n = 100 * env / env.max()                            # % of peak
    rules = {"baseline+3SD": env_n[base].mean() + 3 * env_n[base].std(),
             "10% of peak": 10.0,
             "25% of peak": 25.0}
    for name, thr in rules.items():
        on, off = bursts(env_n, thr)
        print("%5d   %-14s(%4.1f%%)  %d      %s" % (fc, name, thr, len(on), np.round(on, 3)))

# 그림: 처리 단계별 신호 (fc = 6 Hz, baseline+3SD 규칙)
env = envelope(x_rect, 6)
env_n = 100 * env / env.max()
thr = env_n[base].mean() + 3 * env_n[base].std()
on, off = bursts(env_n, thr)
print("fc 6 Hz, baseline+3SD: onset", np.round(on, 3), "offset", np.round(off, 3))
fig, axs = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
axs[0].plot(t, raw, lw=0.5); axs[0].set_ylabel("raw")
axs[1].plot(t, x_rect, lw=0.5, color="gray"); axs[1].set_ylabel("band-pass,\nrectified")
axs[2].plot(t, env_n, "k"); axs[2].axhline(thr, color="r", ls="--", lw=0.8)
for a, b in zip(on, off):
    axs[2].axvspan(a, b, color="r", alpha=0.15)
axs[2].set_ylabel("envelope\n(% of peak)"); axs[2].set_xlabel("time (s)")
out = Path("figures/advanced/I/I5_emg_processing.png")
out.parent.mkdir(parents=True, exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=150)
print("저장:", out)